# Chapter 6 &mdash; Isomorphism = Equivalence + Equal State Count

**Concept 5 of the Chapter 6 decomposition:** *Isomorphism = Language Equivalence + Equal State Count*

For <i>minimal</i> DFA the two checks coincide; for arbitrary DFA they do not.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter6/Concept-Isomorphism-Vs-Equivalence/Concept-Isomorphism-Vs-Equivalence.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


Two crisp facts, often confused:

* **`langeq_dfa`** asks whether the two machines accept the same strings. It works on
  any two DFA over the same alphabet.
* **`iso_dfa`** asks whether they are the same graph up to renaming states.

For **minimal** DFA these coincide: same language $\Rightarrow$ isomorphic
(Myhill&ndash;Nerode, Concept 7). For arbitrary DFA, language equality says nothing about
shape &mdash; you can always pad a machine with unreachable or duplicate states.

Practical rule: **compare languages with `langeq_dfa`; compare designs with
`iso_dfa(min_dfa(A), min_dfa(B))`.**

## 2. Definitions

### One language, two machines of different size

In [ ]:
small = md2mc('''DFA
IF : 0 -> Od
IF : 1 -> IF
Od : 0 -> IF
Od : 1 -> Od
''')
padded = md2mc('''DFA
IF  : 0 -> Od
IF  : 1 -> FDup    !! FDup behaves exactly like IF -- and must be FINAL too
Od  : 0 -> IF
Od  : 1 -> Od
FDup: 0 -> Od
FDup: 1 -> IF
''')

### A helper that reports both verdicts

In [ ]:
def compare(A, B, nameA='A', nameB='B'):
    print("%-22s %s" % ("langeq_dfa", langeq_dfa(A, B)))
    print("%-22s %s" % ("iso_dfa", iso_dfa(A, B)))
    print("%-22s %s" % ("iso after min_dfa", iso_dfa(min_dfa(A), min_dfa(B))))
    print("%-22s %s=%d  %s=%d" % ("|Q|", nameA, len(A["Q"]), nameB, len(B["Q"])))

<!-- nav-strip -->

---

&larr;&nbsp;[Ch6&nbsp;4.&nbsp;Language Equivalence Checking by Lock-Step Search, with Counterexamples](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter6/Concept-Language-Equivalence-Checking/Concept-Language-Equivalence-Checking.ipynb) &nbsp;&middot;&nbsp; [**Chapter 6** index](https://github.com/ganeshutah/Jove/blob/master/Chapter6/README.md) &nbsp;&middot;&nbsp; [Ch6&nbsp;6.&nbsp;The Language Stethoscope: Indistinguishability and Minimality](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter6/Concept-Language-Stethoscope/Concept-Language-Stethoscope.ipynb)&nbsp;&rarr;

---

## 3. Tests

Same language, different shape: `langeq` yes, `iso` no.

In [ ]:
compare(small, padded, 'small', 'padded')
assert langeq_dfa(small, padded)
assert not iso_dfa(small, padded)

But after minimizing, the shapes agree &mdash; as Myhill&ndash;Nerode requires.

In [ ]:
ms, mp = min_dfa(small), min_dfa(padded)
print("minimal sizes : %d and %d" % (len(ms["Q"]), len(mp["Q"])))
assert len(ms["Q"]) == len(mp["Q"])
assert iso_dfa(ms, mp)
print("isomorphic after minimization :", iso_dfa(ms, mp))

Equal state count alone is **not** enough &mdash; both conditions are needed.

In [ ]:
other = md2mc('''DFA
IF : 1 -> Od
IF : 0 -> IF
Od : 1 -> IF
Od : 0 -> Od
''')
print("small vs other: same |Q| (%d) but langeq = %s, iso = %s"
      % (len(other["Q"]), langeq_dfa(small, other), iso_dfa(small, other)))
assert len(small["Q"]) == len(other["Q"]) and not langeq_dfa(small, other)
print("\n'even 0s' and 'even 1s' are different languages with equal-size machines.")

## 4. Exercises


1. Give two minimal DFA that are isomorphic but not *identical*. What differs?
2. Is `iso_dfa` an equivalence relation? Check reflexivity, symmetry, transitivity.
3. Why is `iso_dfa` on non-minimal DFA a *sufficient* but not *necessary* test?

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for all 246 concepts.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter6/Concept-Isomorphism-Vs-Equivalence')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')